# SmartShield: first logistic regression experiment

**Question:** can an explainable model suggest unhandled low-level call results beyond the bounded rules? This is an operation-level pilot, not contract safety prediction. The protocol and held-out split were committed before feature extraction/training. Sources are downloaded unchanged at pinned revisions; no external Solidity source is embedded here.

Run the setup commands in `ml/README.md`, then execute all cells from the repository root. This notebook reuses tested modules and regenerates the audit/results. It never changes the frozen split or ships a model.

In [1]:
from pathlib import Path
import sys, json
ROOT = Path.cwd()
if not (ROOT / "ml").is_dir():
    ROOT = ROOT.parents[1]  # when opened from ml/notebooks
assert (ROOT / "ml/results/config.json").is_file()
sys.path.insert(0, str(ROOT))
from ml.src.data import read_json, compile_batch
from ml.src.audit import run as audit
from ml.src.features import extract, FEATURE_NAMES
from ml.src.experiment import run as experiment
cache = ROOT / ".tools/ml-data"

## 1. Load and audit

SmartBugs broad vulnerability categories and SolidiFI injection locations are candidate annotations, not target ground truth. Both repositories preserve original contract licenses. We commit references and derived metadata, not the downloaded contracts. The audit checks source revision, clean checkout, checksums, original pragma metadata, exact/near duplicates and unchanged product-compiler compatibility.

In [2]:
audit_summary = audit(cache)
print(json.dumps(audit_summary, indent=2))

{'source_files': 493, 'by_candidate': {'smartbugs': 143, 'solidifi': 350}, 'product_compilation': {'compilation_error': 493}, 'exact_duplicate_groups': 7, 'exact_duplicate_excess_files': 7, 'near_duplicate_pairs': 335, 'family_components': 115, 'largest_family_files': 21, 'reviewed_labels': {'verified_negative': 9, 'positive': 8, 'unsupported': 2, 'unknown': 1}, 'unreviewed_files_are': 'unknown; no inferred safe labels', 'split_sha256': 'a893277ea47d58bcd1df0e5d98355fbe279021b57cdff85210a2c918ff759425', 'near_duplicate_threshold': 0.8, 'limitations': ['Normalized token overlap is a conservative leakage screen, not semantic equivalence.', 'Original project identity for some Etherscan examples is unknown.', 'License headers are not legal clearance; raw sources are not redistributed.']}
{
  "source_files": 493,
  "by_candidate": {
    "smartbugs": 143,
    "solidifi": 350
  },
  "product_compilation": {
    "compilation_error": 493
  },
  "exact_duplicate_groups": 7,
  "exact_duplicate_ex

## 2. Define and review labels

Task `UEC-operation-v1`: for a specified compiler-resolved `call`, `delegatecall` or `staticcall`, predict discarded/unhandled success. A negative must visibly check/handle that operation's result. `send`, unreviewed loops and escaping results are excluded or unknown. Other vulnerabilities can coexist with verified negatives. One executing engineer inspected the 20 source operations; no independent human approval is claimed. Selection was purposive and small, not representative. Full rationales and source hashes are in `datasets/labels.json`.

In [3]:
labels = read_json(ROOT / "ml/datasets/labels.json")
from collections import Counter
print(Counter(row["label"] for row in labels))
for row in labels[:2]:
    print(row["source_id"], row["line"], row["label"], row["rationale"])

Counter({'verified_negative': 9, 'positive': 8, 'unsupported': 2, 'unknown': 1})
smartbugs/dataset/unchecked_low_level_calls/unchecked_return_value.sol 12 verified_negative require directly consumes the success bool and reverts on false.
smartbugs/dataset/unchecked_low_level_calls/unchecked_return_value.sol 17 positive Standalone call expression discards success.


## 3. Frozen project-separated split

Known original projects, exact duplicates, normalized-token near duplicates and all injected variants of a SolidiFI base contract are unioned before the deterministic 25% family-hash holdout. Seed/split were not retried for class balance. Acceptance fixtures are excluded. Token matching is a leakage screen, never a vulnerability detector. Unknown Etherscan provenance and undetected semantic relatives remain limitations.

In [4]:
split = read_json(ROOT / "ml/results/split.json")
print("Frozen split:", audit_summary["split_sha256"])
for part in ["train", "test"]:
    rows = [r for r in labels if split[r["source_id"]]["partition"] == part]
    print(part, Counter(r["label"] for r in rows),
          "families", len({split[r["source_id"]]["family"] for r in rows}))

Frozen split: a893277ea47d58bcd1df0e5d98355fbe279021b57cdff85210a2c918ff759425
train Counter({'verified_negative': 7, 'positive': 4, 'unsupported': 1}) families 9
test Counter({'positive': 4, 'verified_negative': 2, 'unsupported': 1, 'unknown': 1}) families 5


## 4. Explainable compiler features

The shared extractor accepts typed compiler ASTs. Features describe result context, declaration-linked uses, guard/conditional ancestors, call kind and small statement counts. They exclude names, comments, filenames, benchmark categories and rule findings/statuses. No regex determines predictions or labels. For the historical pilot only, unchanged reviewed contracts are compiled with pinned solc 0.4.25. This does not extend the product API's compiler support.

In [5]:
example = labels[0]
source = (cache / "smartbugs" / example["source_id"].removeprefix("smartbugs/")).read_text()
compiled = compile_batch([{"id": "example", "source": source, "ast": True}], legacy=True)[0]
assert compiled["status"] == "compiled"
records = extract(compiled["ast"], source)
print("Feature schema:", FEATURE_NAMES)
print([(r["span"], r.get("features"), r["status"]) for r in records])

Feature schema: ('standalone_expression', 'assigned_result', 'later_result_references', 'guard_ancestor', 'condition_ancestor', 'unary_ancestor', 'delegatecall', 'staticcall', 'value_option', 'argument_count', 'function_statement_count')
[({'offset': 263, 'length': 13, 'line': 12, 'endLine': 12}, {'standalone_expression': 0, 'assigned_result': 0, 'later_result_references': 0, 'guard_ancestor': 1, 'condition_ancestor': 0, 'unary_ancestor': 0, 'delegatecall': 0, 'staticcall': 0, 'value_option': 0, 'argument_count': 0, 'function_statement_count': 1}, 'supported'), ({'offset': 381, 'length': 13, 'line': 17, 'endLine': 17}, {'standalone_expression': 1, 'assigned_result': 0, 'later_result_references': 0, 'guard_ancestor': 0, 'condition_ancestor': 0, 'unary_ancestor': 0, 'delegatecall': 0, 'staticcall': 0, 'value_option': 0, 'argument_count': 0, 'function_statement_count': 1}, 'supported')]


## 5. Train one fixed baseline and evaluate once

Use L2 logistic regression, C=1, liblinear, seed 2026 and threshold 0.5. Scaling fits training data only. No class weighting, tuning, probability calibration or second model: this sample cannot justify them. The experiment records exclusions before fitting and uses the unchanged C++ analyzer as a separate historical AST probe. Product-domain metrics are separate and undefined when no contracts compile. Timing covers inference only; serialized size is measured in memory and no predictor is saved.

In [6]:
import contextlib, io
with contextlib.redirect_stdout(io.StringIO()):
    result = experiment(cache)
print(result["counts"])
print(json.dumps(result["historical_pilot"], indent=2))
print("Matched C++ baseline:", result["historical_cpp_baseline_completed_only"])
print("Baseline statuses:", result["historical_baseline_statuses"])
print("Product coverage:", result["product_coverage"])
print("Timing/size:", result["measurement"])

{'reviewed': 20, 'train': 11, 'historical_test': 6, 'historical_test_families': 4, 'train_classes': {'verified_negative': 7, 'positive': 4}, 'exclusions': {'Reviewed unsupported': 2, 'Reviewed unknown': 1}, 'product_test': 0}
{
  "n": 6,
  "confusion_matrix": [
    [
      2,
      0
    ],
    [
      0,
      4
    ]
  ],
  "matrix_order": [
    "verified_negative",
    "positive"
  ],
  "per_class": {
    "verified_negative": {
      "precision": 1.0,
      "recall": 1.0,
      "f1": 1.0,
      "support": 2,
      "predicted_count": 2,
      "precision_95_wilson": [
        0.34238022750665303,
        1.0
      ],
      "recall_95_wilson": [
        0.34238022750665303,
        1.0
      ]
    },
    "positive": {
      "precision": 1.0,
      "recall": 1.0,
      "f1": 1.0,
      "support": 4,
      "predicted_count": 4,
      "precision_95_wilson": [
        0.5101091635454027,
        1.0
      ],
      "recall_95_wilson": [
        0.5101091635454027,
        1.0
      ]
    }


## 6. Inspect mistakes and uncertainty

The six held-out historical operations have no classification errors, but only four families are represented. The two proxy variants are related. Centra's discarded first call is close to the fixed 0.5 threshold; its handled second call is a distinct negative on the same contract. A perfect tiny score is not persuasive evidence of generalization. Per-operation Wilson intervals are descriptive; correlation within families makes them insufficient for a deployment claim. Broad access-control/reentrancy labels do not make checked-call examples positive for this task; discarded `send` examples are out of scope.

In [7]:
print("Misclassifications:", read_json(ROOT / "ml/results/errors.json"))
operations = read_json(ROOT / "ml/results/operations.json")
for row in operations:
    if "prediction" in row:
        print(row["source_id"].split("/")[-1], row["line"], row["label"],
              round(row["prediction"]["score"], 3), row["baseline"]["status"])

Misclassifications: []
etherbank.sol 21 verified_negative 0.122 unsupported
0x0cbe050f75bc8f8c2d6c0d249fea125fd6e1acc9.sol 12 positive 0.9 completed
0x524960d55174d912768678d8c606b4d50b79d7b1.sol 21 positive 0.526 unsupported
0x524960d55174d912768678d8c606b4d50b79d7b1.sol 22 verified_negative 0.01 unsupported
0xb11b2fed6c9354f7aa2f658d3b4d7b31d8a13b77.sol 14 positive 0.921 unsupported
0xbaa3de6504690efb064420d89e871c27065cdd52.sol 14 positive 0.921 unsupported


## 7. Integration decision

No integration. All 493 external files fail unchanged product compilation; eligible product test denominator is zero. The historical probe has only one completed-rule comparison, on which both methods agree. Unsupported rule outcomes are abstentions, never evidence of ML improvement. The preregistered sample-size, coverage, uncertainty and incremental-value requirements fail. The four rule statuses stay authoritative; there is no model artifact, API/schema change or cosmetic ML panel.

Next experiment needs independently reviewed modern Solidity operations across at least 20 held-out project families, meaningful assigned-result/branch cases, and matched completed-rule baseline comparisons. More injected near-copies cannot supply independent evidence.

In [8]:
print(json.dumps(result["integration"], indent=2))
assert result["integration"]["integrate"] is False
assert result["measurement"]["artifact_shipped"] is False

{
  "integrate": false,
  "reasons": [
    "Too few independently grouped eligible test families",
    "Insufficient eligible held-out cases per class",
    "Product compiler/feature coverage below gate",
    "Positive precision uncertainty does not satisfy gate",
    "No demonstrated incremental value over matched completed-rule baseline"
  ]
}
